# Experimento — STab

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

O STab de Hajiramezanali, Shen, Scalia e Diamant (2022) aprende uma representação da tabela sem aumento de dados na entrada. Dois codificadores MLP compartilham os pesos, cada um com uma regularização estocástica diferente, e uma cabeça de projeção alinha as duas saídas com stop-gradient. Depois desse estágio, uma cabeça supervisionada classifica `Churn`.


## Sobre o nome

O pedido agrupou este experimento como modelo baseado em Transformer. O STab publicado é o par de MLPs descrito acima, sem camada de atenção. O roteiro segue esse artigo. Se o alvo for outro modelo, este notebook troca de nome antes do bloco 0.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

O artigo fixa o método e publica a arquitetura usada na tabela de resultados. Não publica o p da máscara, o otimizador, a taxa, as épocas nem o batch, e não há repositório oficial. Onde o número não existe, o bloco 0 anota a escolha inicial.

Ficam fora da busca, porque são o método: dois codificadores, pesos compartilhados, stop-gradient e perda de cosseno simétrica, sem temperatura. Sem o stop-gradient a solução trivial aparece.

A arquitetura publicada é o ponto de partida. O codificador tem 4 camadas, [256, 256, 256, 256]. A cabeça de projeção g tem 2 camadas, [256, 256]. A ativação publicada é ReLU. Profundidade do codificador, largura do codificador, profundidade de g e largura de g são quatro blocos. Quando a saída do codificador muda, a entrada de g acompanha; isso não conta como segundo knob.

Logística e tangente hiperbólica entram depois, na rede já aceita, contra a ReLU do artigo.

DropConnect é a máscara nos pesos, sorteada por amostra. No artigo, p é a probabilidade de a máscara valer 1, como em Wan et al. (2013): a conexão cai com probabilidade 1−p. Dropout é a máscara nas unidades. Essa é a única comparação que o artigo mede, e o DropConnect fica acima nos quatro conjuntos. O valor de p não é publicado. Um único p vale para as duas vistas e para todas as camadas. A regularização L2 não aparece no artigo e só fica se um bloco com ela vencer o que já foi aceito.

A cabeça supervisionada do artigo é regressão logística sobre o embedding congelado. Uma MLP no lugar dela é um bloco, com o pré-treino parado, e só fica se vencer essa regressão.

O artigo diz que os pesos podem ser atualizados com SGD ou Adam, e não diz qual dos dois gerou a tabela. L-BFGS entra na comparação pedida para as redes deste projeto. A taxa é o bloco seguinte. A taxa adaptativa só entra se o SGD vencer. Batch e épocas de pré-treino são blocos próprios: a perda de cosseno não depende do tamanho do lote, o número de passos depende. Se as épocas forem cortadas por early stopping, a paciência é mais um bloco. Nenhum desses números está no artigo.

O oversampling repetitivo das dobras vale para a cabeça supervisionada. O pré-treino sem rótulo usa as linhas de treino da dobra sem duplicar pela classe.

Fonte: o PDF do OpenReview, https://openreview.net/pdf?id=EfR55bFcrcI. O apêndice não traz tabela de hiperparâmetros.


## Hiperparâmetros que ainda vão chegar

A pesquisa destes hiperparâmetros está nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. Arquitetura da tabela do artigo, ReLU, DropConnect, dois codificadores, stop-gradient, cosseno e regressão logística sobre o embedding congelado. p, taxa, batch e épocas ficam numa escolha inicial anotada.
2. Dropout no lugar do DropConnect, com o mesmo p e a mesma rede.
3. O p único.
4. Profundidade do codificador.
5. Largura do codificador.
6. Profundidade de g, depois a largura de g.
7. Logística e tangente hiperbólica, contra a ReLU já aceita.
8. Cabeça MLP, se for aberta, contra a regressão logística.
9. SGD, Adam e L-BFGS.
10. Taxa de aprendizagem. A taxa adaptativa só entra se o SGD tiver vencido.
11. Tamanho do lote.
12. Épocas de pré-treino. Se houver early stopping, a paciência é o bloco seguinte.
13. L2, só se for aberta.
14. Estratégias de dados compartilhadas.
15. Bloco final no teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
